# **Creación de dataset EEG**

## Lectura de los datos EMG

In [1]:
import numpy as np
import pandas as pd
import re

def leertxt(filename:str,header:int=2,delimiter:str=',',media:bool=True) -> tuple[np.ndarray, float]:
    # Abrimos el archivo txt para ver  el contenido
    f = open(filename,"r")
    raw_data = f.readline()  # con f.read() leemos todo el contenido
    f.close()

    ## Expresion regular para buscar automaticamente el contenido de un numero dentro de un string
    x = re.findall("[0-5][0-9]\d", raw_data)
    Fs = float(x[0])

    # Leemos el archivo excluyendo las dos primeras filas del archivo
    array = np.genfromtxt("./"+filename, delimiter=delimiter,skip_header = header)
    if media: array[:,1:]= array[:,1:] - np.mean(array[:,1])    #le restamos la media

    return array,Fs

array1,Fs=leertxt('señal bitalino 1.txt')
array2=leertxt('señal bitalino 2.txt')[0]
array3,Fs3=leertxt('señal ultracortex.txt',4,media=False)

array1[:,0] = np.arange(0, len(array1[:,0]))
basal1 = array1[0:3000,1]
ojos = array1[3000:6900,1]
basal2 = array1[6900:9900,1]
p1 = array1[9900:11900,1]
p2 = array1[11900:13100,1]
p3 = array1[13100:14300,1]
p4 = array1[14300:16300,1]
p5 = array1[16300:18500,1]
p6 = array1[18500:20600,1]
p7 = array1[20600:22800,1]

array2[:,0] = np.arange(0, len(array2[:,0]))
ciegas = array2[0:6000,1]
luces = array2[6000:12000,1]

ch0=array3[:14200,1]
ch1=array3[:14200,2]
ch2=array3[:14200,3]
ch3=array3[:14200,4]
ch4=array3[:14200,5]
ch5=array3[:14200,6]
ch6=array3[:14200,7]
ch7=array3[:14200,8]

## Pasando a data tabular estilo Sklearn

In [2]:
def deefe(array:np.ndarray,ntarget:int,col:int=100,Fs:int=Fs) -> tuple[np.ndarray, np.ndarray]:
  Ts=1/Fs
  n = np.arange(0,array.shape[0])  # t = n*Ts
  t = n*Ts

  #Pasamos las observaciones a filas correspondientes a una variable t y d_sensor
  st_sensor = np.concatenate((t.reshape(-1,1),  array.reshape(-1,1)), axis=1)
  #Creamos el data frame con las varibles t y d_sensor
  df = pd.DataFrame(st_sensor, columns=["t","d_sensor"])
  df.head()
  #Establecemos t como index 
  df = df.set_index("t")

  d_obs = df[["d_sensor"]].values.reshape(int(array.shape[0]/col),col)
  target = np.repeat(ntarget, d_obs.shape[0])
  return d_obs,target

obs_basal1,target_basal1=deefe(basal1,0)
obs_ojos,target_ojos=deefe(ojos,1)
obs_basal2,target_basal2=deefe(basal2,2)
obs_p1,target_p1=deefe(p1,3)
obs_p2,target_p2=deefe(p2,4)
obs_p3,target_p3=deefe(p3,5)
obs_p4,target_p4=deefe(p4,6)
obs_p5,target_p5=deefe(p5,7)
obs_p6,target_p6=deefe(p6,8)
obs_p7,target_p7=deefe(p7,9)

obs_ciegas,target_ciegas=deefe(ciegas,10)
obs_luces,target_luces=deefe(luces,11)

obs_ch0,target_ch0=deefe(ch0,12,Fs=Fs3)
obs_ch1,target_ch1=deefe(ch1,13,Fs=Fs3)
obs_ch2,target_ch2=deefe(ch2,14,Fs=Fs3)
obs_ch3,target_ch3=deefe(ch3,15,Fs=Fs3)
obs_ch4,target_ch4=deefe(ch4,16,Fs=Fs3)
obs_ch5,target_ch5=deefe(ch5,17,Fs=Fs3)
obs_ch6,target_ch6=deefe(ch6,18,Fs=Fs3)
obs_ch7,target_ch7=deefe(ch7,19,Fs=Fs3)

## Descripción de categoría de los ejercicios realizado en la clase de ECG

| Descripción | Categoría |
|----------|----------|
|Estado basal 1 (bitalino)                  |0|
|Abriendo y cerrando los ojos (bitalino)    |1|
|Estado basal 2 (bitalino)                  |2|
|Pregunta 1 (bitalino)                      |3|
|Pregunta 2 (bitalino)                      |4|
|Pregunta 3 (bitalino)                      |5|
|Pregunta 4 (bitalino)                      |6|
|Pregunta 5 (bitalino)                      |7|
|Pregunta 6 (bitalino)                      |8|
|Pregunta 7 (bitalino)                      |9|
|A ciegas (bitalino)                        |10|
|Con luces (bitalino)                       |11|
|Canal 0 (Ultracortex)                      |12|
|Canal 1 (Ultracortex)                      |13|
|Canal 2 (Ultracortex)                      |14|
|Canal 3 (Ultracortex)                      |15|
|Canal 4 (Ultracortex)                      |16|
|Canal 5 (Ultracortex)                      |17|
|Canal 6 (Ultracortex)                      |18|
|Canal 7 (Ultracortex)                      |19|

## Creación del diccionario

In [3]:
eeg = {"base": np.concatenate([obs_basal1,obs_ojos,obs_basal2,obs_p1,obs_p2,obs_p3,obs_p4,obs_p5,obs_p6,obs_p7,obs_ciegas,obs_luces,obs_ch0,obs_ch1,obs_ch2,obs_ch3,obs_ch4,obs_ch5,obs_ch6,obs_ch7]), "target": np.concatenate([target_basal1,target_ojos,target_basal2,target_p1,target_p2,target_p3,target_p4,target_p5,target_p6,target_p7,target_ciegas,target_luces,target_ch0,target_ch1,target_ch2,target_ch3,target_ch4,target_ch5,target_ch6,target_ch7])}
eeg

{'base': array([[ 3.29584155e+02,  3.86584155e+02,  4.04584155e+02, ...,
          6.58415528e+00, -1.74158447e+01, -3.44158447e+01],
        [ 3.58415528e+00, -4.41584472e+00,  2.58415528e+00, ...,
         -1.41584472e+00, -1.94158447e+01, -8.41584472e+00],
        [-3.84158447e+01,  3.65841553e+01,  1.35841553e+01, ...,
         -3.84158447e+01,  5.65841553e+01, -8.41584472e+00],
        ...,
        [-1.55917678e+05,  3.32194532e+04, -1.55896734e+05, ...,
         -1.55968059e+05,  3.23013106e+04, -1.55989047e+05],
        [ 3.19476166e+04, -1.55983839e+05,  3.22735721e+04, ...,
          3.42178386e+04, -1.55888218e+05,  3.37474684e+04],
        [-1.55890565e+05,  3.41722857e+04, -1.55861486e+05, ...,
         -1.55283917e+05,  3.25493479e+04, -1.55297238e+05]]),
 'target': array([ 0,  0,  0, ..., 19, 19, 19])}

## Exportar el diccionario a un archivo .pkl

In [4]:
import pickle
with open('dataset_EEG.pkl', 'wb') as f:
    pickle.dump(eeg, f)